#Two-Stage Reranking Workflow

1. Stage 1 → BM25 candidate retrieval

   - BM25 is very fast → grab top-k docs quickly (e.g., top 20).

2. Stage 2 → Semantic reranking

   - Re-rank only those top-k docs using embeddings (cosine similarity).

   - Much faster than computing embeddings for the entire corpus.

In [3]:
pip install rank-bm25 sentence-transformers

In [4]:
# imports
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

In [5]:
# -----------------------------
# 1. Corpus & Query
# -----------------------------
corpus = [
    "The car engine needs repair",
    "Automobile motor fixing guide",
    "Learn how to bake a chocolate cake",
    "Best practices for vehicle maintenance",
    "Python tutorial for beginners",
    "Engine troubleshooting for trucks",
    "Cake decoration tips and tricks"
]
query = "How to fix a vehicle engine?"

In [6]:
# -----------------------------
# 2. BM25 Retrieval (Stage 1)
# -----------------------------
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

In [7]:
bm25_scores = bm25.get_scores(query.lower().split())

In [8]:
# Pick top-k candidates
k = 3
top_k_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
bm25_candidates = [corpus[i] for i in top_k_idx]

In [9]:
print("\nBM25 Top-k Candidates:")
for doc in bm25_candidates:
    print(" -", doc)


BM25 Top-k Candidates:
 - Learn how to bake a chocolate cake
 - Best practices for vehicle maintenance
 - The car engine needs repair


In [11]:
# -----------------------------
# 3. Embedding Reranking (Stage 2)
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

query_emb = model.encode(query, convert_to_tensor=True)
cand_embs = model.encode(bm25_candidates, convert_to_tensor=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
# Compute semantic similarity
cosine_scores = util.cos_sim(query_emb, cand_embs)[0].cpu().numpy()

# Sort by embedding score
reranked = sorted(zip(bm25_candidates, cosine_scores), key=lambda x: x[1], reverse=True)


In [13]:
# -----------------------------
# 4. Final Ranked Results
# -----------------------------
print("\nFinal Reranked Results:")
for doc, score in reranked:
    print(f"Doc: {doc}\n  Semantic Score: {score:.3f}\n")


Final Reranked Results:
Doc: The car engine needs repair
  Semantic Score: 0.758

Doc: Best practices for vehicle maintenance
  Semantic Score: 0.386

Doc: Learn how to bake a chocolate cake
  Semantic Score: 0.096



In [15]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.0 MB/s eta 0:00:00


In [16]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# -----------------------------
# 1. Corpus & Query
# -----------------------------
corpus = [
    "The car engine needs repair",
    "Automobile motor fixing guide",
    "Learn how to bake a chocolate cake",
    "Best practices for vehicle maintenance",
    "Python tutorial for beginners",
    "Engine troubleshooting for trucks"
]
query = "How to fix a vehicle engine?"

# -----------------------------
# 2. BM25 Retrieval
# -----------------------------
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.lower().split())

# -----------------------------
# 3. Embeddings + FAISS
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = model.encode(corpus, convert_to_numpy=True)

# Build FAISS index
d = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(d)  # cosine similarity (inner product)
faiss.normalize_L2(corpus_embeddings)  # normalize for cosine
index.add(corpus_embeddings)

# Query embedding
query_emb = model.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_emb)

# Search top-k
k = 3
D, I = index.search(query_emb, k)

faiss_scores = np.zeros(len(corpus))
for score, idx in zip(D[0], I[0]):
    faiss_scores[idx] = score

# -----------------------------
# 4. Hybrid Scoring
# -----------------------------
def normalize(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x) + 1e-9)

bm25_norm = normalize(bm25_scores)
faiss_norm = normalize(faiss_scores)

alpha = 0.5
hybrid_scores = alpha * bm25_norm + (1 - alpha) * faiss_norm

# Final ranking
results = sorted(
    zip(corpus, bm25_norm, faiss_norm, hybrid_scores),
    key=lambda x: x[3], reverse=True
)

print("\nFinal Hybrid Results:")
for doc, bm25_s, faiss_s, hyb_s in results:
    print(f"Doc: {doc}\n  BM25: {bm25_s:.3f}, FAISS: {faiss_s:.3f}, Hybrid: {hyb_s:.3f}\n")


Final Hybrid Results:
Doc: Learn how to bake a chocolate cake
  BM25: 1.000, FAISS: 0.000, Hybrid: 0.500

Doc: The car engine needs repair
  BM25: 0.000, FAISS: 1.000, Hybrid: 0.500

Doc: Automobile motor fixing guide
  BM25: 0.000, FAISS: 0.795, Hybrid: 0.398

Doc: Engine troubleshooting for trucks
  BM25: 0.000, FAISS: 0.668, Hybrid: 0.334

Doc: Best practices for vehicle maintenance
  BM25: 0.394, FAISS: 0.000, Hybrid: 0.197

Doc: Python tutorial for beginners
  BM25: 0.000, FAISS: 0.000, Hybrid: 0.000

